# Валидация

Проверяем что staging и fact сошлись по количеству строк и нет пустых ключей.

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5433,
    database="bdsnowflake",
    user="bds_user",
    password="bds_password",
)

## Количество строк

In [ ]:
rows_df = pd.read_sql(
    """
    SELECT 'raw.mock_data_staging' AS table_name, count(*) AS row_count FROM raw.mock_data_staging
    UNION ALL
    SELECT 'dw.fact_sales' AS table_name, count(*) AS row_count FROM dw.fact_sales
    """,
    conn,
)
rows_df

## NULL-ключи в факте

In [ ]:
fk_df = pd.read_sql(
    """
    SELECT
      sum(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer_key,
      sum(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS null_product_key,
      sum(CASE WHEN date_key IS NULL THEN 1 ELSE 0 END) AS null_date_key
    FROM dw.fact_sales
    """,
    conn,
)
fk_df

In [ ]:
staging_rows = int(rows_df.loc[rows_df["table_name"] == "raw.mock_data_staging", "row_count"].iloc[0])
fact_rows = int(rows_df.loc[rows_df["table_name"] == "dw.fact_sales", "row_count"].iloc[0])
null_fk_total = int(fk_df.sum(axis=1).iloc[0])

print(f"staging:  {staging_rows}")
print(f"fact:     {fact_rows}")
print(f"null FK:  {null_fk_total}")

if staging_rows == fact_rows and null_fk_total == 0:
    print("\nвсё ок")
else:
    print("\nчто-то не так, надо смотреть")

In [ ]:
conn.close()